# Film Clustering — Content Only (No Country/Language Features)This notebook is a modified version of the original clustering notebook. The key change is that **country and language metadata are excluded from the clustering features**, so clusters are defined purely by narrative content (plot text), genres, and keywords.This ensures that the resulting clusters reflect *what a film is about* rather than *where it was made*, which is essential for the downstream predictor: we want to ask "how does content type X perform in country Y?" without country already being baked into the cluster definition.

In [ ]:
import osos.environ["OMP_NUM_THREADS"] = "1"os.environ["MKL_NUM_THREADS"] = "1"os.environ["OPENBLAS_NUM_THREADS"] = "1"os.environ["NUMEXPR_NUM_THREADS"] = "1" 

In [ ]:
from pathlib import Pathfrom collections import Counterimport astimport joblibimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy.sparse import hstack, load_npzfrom sklearn.cluster import KMeans, MiniBatchKMeansfrom sklearn.decomposition import TruncatedSVDfrom sklearn.metrics import silhouette_scorefrom sklearn.preprocessing import normalizefrom threadpoolctl import threadpool_limits

## 1. Load the saved feature artifactsWe load all feature matrices from notebook 01, including the metadata matrix. However, **the metadata matrix will only be used for profiling and interpretation**, not as a clustering input.

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "text_clustering"FEATURES_DIR = OUTPUTS_DIR / "features"RESULTS_DIR = OUTPUTS_DIR / "results_content_only"OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)FEATURES_DIR.mkdir(parents=True, exist_ok=True)RESULTS_DIR.mkdir(parents=True, exist_ok=True)PROCESSED_PATH = FEATURES_DIR / "processed_films.csv"TFIDF_VECTORIZER_PATH = FEATURES_DIR / "tfidf_vectorizer.joblib"GENRE_MLB_PATH = FEATURES_DIR / "genre_mlb.joblib"KEYWORD_MLB_PATH = FEATURES_DIR / "keyword_mlb.joblib"METADATA_OHE_PATH = FEATURES_DIR / "metadata_ohe.joblib"X_PLOT_TFIDF_PATH = FEATURES_DIR / "X_plot_tfidf.npz"X_GENRES_PATH = FEATURES_DIR / "X_genres.npz"X_KEYWORDS_PATH = FEATURES_DIR / "X_keywords.npz"X_METADATA_PATH = FEATURES_DIR / "X_metadata.npz"X_PLOT_ONLY_PATH = FEATURES_DIR / "X_plot_only.npz"X_FULL_BASELINE_PATH = FEATURES_DIR / "X_full_baseline.npz" 

In [ ]:
df = pd.read_csv(PROCESSED_PATH)tfidf_vectorizer = joblib.load(TFIDF_VECTORIZER_PATH)genre_mlb = joblib.load(GENRE_MLB_PATH)keyword_mlb = joblib.load(KEYWORD_MLB_PATH)metadata_ohe = joblib.load(METADATA_OHE_PATH)X_plot_tfidf = load_npz(X_PLOT_TFIDF_PATH)X_genres = load_npz(X_GENRES_PATH)X_keywords = load_npz(X_KEYWORDS_PATH)X_metadata = load_npz(X_METADATA_PATH)X_plot_only = load_npz(X_PLOT_ONLY_PATH)X_full_baseline = load_npz(X_FULL_BASELINE_PATH)print("DataFrame shape:", df.shape)print("X_plot_tfidf:", X_plot_tfidf.shape)print("X_genres:", X_genres.shape)print("X_keywords:", X_keywords.shape)print("X_metadata:", X_metadata.shape, "(loaded for profiling only, NOT used in clustering)")print("X_plot_only:", X_plot_only.shape)print("X_full_baseline:", X_full_baseline.shape)

In [ ]:
if "title" in df.columns:    display_col = "title"else:    display_col = df.columns[0]print("Display column:", display_col)df[[display_col]].head()

## 2. Construct the content-only clustering representationUnlike the original notebook, we **exclude the metadata block** (country and language) from the clustering matrix. The representation now combines only:- TF-IDF on plot text (weight 0.85) — the primary signal- Multi-hot genres (weight 0.55) — editorial genre labels- Filtered keywords (weight 0.55) — curated thematic tagsCountry and language are preserved in the dataframe for later profiling, but they do not influence which cluster a film belongs to.

In [ ]:
PLOT_WEIGHT = 0.85GENRE_WEIGHT = 0.55KEYWORD_WEIGHT = 0.55# Normalize each block independently before weighting.X_plot_block = normalize(X_plot_tfidf, norm="l2", axis=1)X_genres_block = normalize(X_genres, norm="l2", axis=1)X_keywords_block = normalize(X_keywords, norm="l2", axis=1)# Content-only hybrid: NO metadata block.X_content_only = hstack([    X_plot_block * PLOT_WEIGHT,    X_genres_block * GENRE_WEIGHT,    X_keywords_block * KEYWORD_WEIGHT]).tocsr()print("Content-only hybrid matrix:", X_content_only.shape)print("(Original hybrid with metadata would have been wider by", X_metadata.shape[1], "columns)")

In [ ]:
def prepare_matrix_for_clustering(matrix, n_components=150, random_state=42):    X_norm = normalize(matrix, norm="l2", axis=1)    max_components = min(X_norm.shape[0] - 1, X_norm.shape[1] - 1, n_components)    if max_components < 2:        raise ValueError("Not enough dimensions for TruncatedSVD.")    svd = TruncatedSVD(n_components=max_components, random_state=random_state)    X_reduced = svd.fit_transform(X_norm)    X_reduced = normalize(X_reduced, norm="l2")    explained_var = float(svd.explained_variance_ratio_.sum())    return X_reduced, svd, explained_var

## 3. Evaluate candidate clustering solutionsWe evaluate K-Means for k = 4 through 8 on the content-only representation, using the same evaluation protocol as the original notebook: silhouette score, imbalance ratio, and inertia on a sample of 400 films.

In [ ]:
def evaluate_kmeans_solutions(    matrix,    k_values,    random_state=42,    n_init=3,    n_components=30,    eval_sample_size=400,    silhouette_sample_size=200):    rng = np.random.default_rng(random_state)    n_rows = matrix.shape[0]    sample_n = min(eval_sample_size, n_rows)    sampled_idx = np.sort(rng.choice(n_rows, size=sample_n, replace=False))    matrix_sample = matrix[sampled_idx]    with threadpool_limits(limits=1):        X_for_clustering, svd_model, explained_var = prepare_matrix_for_clustering(            matrix_sample,            n_components=n_components,            random_state=random_state        )    results = []    with threadpool_limits(limits=1):        for k in k_values:            model = MiniBatchKMeans(                n_clusters=k,                random_state=random_state,                n_init=n_init,                batch_size=256,                max_iter=100,                reassignment_ratio=0.01            )            labels = model.fit_predict(X_for_clustering)            silhouette = silhouette_score(                X_for_clustering,                labels,                sample_size=min(silhouette_sample_size, len(labels)),                random_state=random_state            )            cluster_sizes = pd.Series(labels).value_counts().sort_index()            results.append({                "k": k,                "silhouette_score": float(silhouette),                "min_cluster_size": int(cluster_sizes.min()),                "max_cluster_size": int(cluster_sizes.max()),                "mean_cluster_size": float(cluster_sizes.mean()),                "imbalance_ratio": float(cluster_sizes.max() / cluster_sizes.min()),                "svd_explained_variance": float(explained_var),                "evaluation_rows_used": int(sample_n),                "inertia": float(model.inertia_)            })    return pd.DataFrame(results).sort_values("k").reset_index(drop=True)

In [ ]:
K_VALUES = [4, 5, 6, 7, 8]evaluation_content_only = evaluate_kmeans_solutions(    X_content_only,    K_VALUES,    n_init=3,    n_components=30,    eval_sample_size=400,    silhouette_sample_size=200)display(    evaluation_content_only[        ["k", "silhouette_score", "imbalance_ratio", "min_cluster_size", "max_cluster_size", "inertia"]    ])

In [ ]:
eval_plot = evaluation_content_only.sort_values("k")fig, axes = plt.subplots(1, 3, figsize=(18, 4))axes[0].plot(eval_plot["k"], eval_plot["silhouette_score"], marker="o")axes[0].set_title("Silhouette score by k")axes[0].set_xlabel("k")axes[0].set_ylabel("Silhouette score")axes[0].set_xticks(eval_plot["k"])axes[0].grid(True)axes[1].plot(eval_plot["k"], eval_plot["imbalance_ratio"], marker="o")axes[1].set_title("Imbalance ratio by k")axes[1].set_xlabel("k")axes[1].set_ylabel("Imbalance ratio")axes[1].set_xticks(eval_plot["k"])axes[1].grid(True)axes[2].plot(eval_plot["k"], eval_plot["inertia"], marker="o")axes[2].set_title("Elbow plot")axes[2].set_xlabel("k")axes[2].set_ylabel("Inertia")axes[2].set_xticks(eval_plot["k"])axes[2].grid(True)plt.tight_layout()plt.show()

## 4. Compare the final candidate solutionsAs with the original analysis, we retain k=5 and k=6 as the most relevant candidates for full inspection. The evaluation metrics above may yield different preferences now that country/language are removed — inspect the plots before deciding.We run the full clustering on both candidates and compare their size distributions.

In [ ]:
def run_full_clustering(matrix, base_df, k, n_components=150, random_state=42, n_init=10):    X_for_clustering, svd_model, explained_var = prepare_matrix_for_clustering(        matrix,        n_components=n_components,        random_state=random_state    )    model = KMeans(        n_clusters=k,        random_state=random_state,        n_init=n_init,        max_iter=300,        algorithm="elkan"    )    with threadpool_limits(limits=1):        labels = model.fit_predict(X_for_clustering)    df_result = base_df.copy()    df_result["cluster"] = labels    cluster_sizes = (        df_result["cluster"]        .value_counts()        .sort_index()        .rename_axis("cluster")        .reset_index(name="size")    )    return {        "k": k,        "model": model,        "df": df_result,        "labels": labels,        "X_for_clustering": X_for_clustering,        "svd_explained_variance": explained_var,        "cluster_sizes": cluster_sizes    }

In [ ]:
run_k5 = run_full_clustering(X_content_only, df, k=5)run_k6 = run_full_clustering(X_content_only, df, k=6)print("k=5 cluster sizes")display(run_k5["cluster_sizes"])print("k=6 cluster sizes")display(run_k6["cluster_sizes"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))axes[0].bar(run_k5["cluster_sizes"]["cluster"].astype(str), run_k5["cluster_sizes"]["size"])axes[0].set_title("Cluster sizes for k=5")axes[0].set_xlabel("Cluster")axes[0].set_ylabel("Number of films")axes[1].bar(run_k6["cluster_sizes"]["cluster"].astype(str), run_k6["cluster_sizes"]["size"])axes[1].set_title("Cluster sizes for k=6")axes[1].set_xlabel("Cluster")axes[1].set_ylabel("Number of films")plt.tight_layout()plt.show()

## 5. Build cluster profilesWe still report the dominant country and language for each cluster — this is now purely *descriptive* rather than a clustering input. This means we can observe whether content-based clusters happen to correlate with geography (which would be an interesting finding in itself) without that correlation being an artifact of the feature design.

In [ ]:
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

In [ ]:
def get_representative_films(matrix, model, labels, cluster_id, df, title_col, top_n=5):    cluster_indices = np.where(labels == cluster_id)[0]    if len(cluster_indices) == 0:        return pd.DataFrame(columns=[title_col])    X_cluster = np.asarray(matrix[cluster_indices])    centroid = np.asarray(model.cluster_centers_[cluster_id]).ravel()    distances = ((X_cluster - centroid) ** 2).sum(axis=1)    closest_local_idx = np.argsort(distances)[:top_n]    closest_global_idx = cluster_indices[closest_local_idx]    return df.iloc[closest_global_idx][[title_col]].reset_index(drop=True)

In [ ]:
def safe_parse_list(value):    if isinstance(value, list):        return value    if pd.isna(value):        return []    if isinstance(value, str):        value = value.strip()        if not value:            return []        try:            parsed = ast.literal_eval(value)            if isinstance(parsed, list):                return parsed        except Exception:            pass        return [item.strip() for item in value.split(",") if item.strip()]    return []def get_existing_column(df, candidates, default=None):    for col in candidates:        if col in df.columns:            return col    return defaultdef top_terms_from_cluster_mean(X_tfidf, labels, cluster_id, feature_names, top_n=8):    cluster_rows = X_tfidf[labels == cluster_id]    mean_vector = np.asarray(cluster_rows.mean(axis=0)).ravel()    top_idx = np.argsort(mean_vector)[::-1][:top_n]    return [feature_names[i] for i in top_idx]def build_cluster_profiles(    df_clustered,    model,    matrix_for_clustering,    top_terms_n=8,    top_genres_n=5,    top_keywords_n=5,    top_representatives_n=5):    labels = df_clustered["cluster"].to_numpy()    profiles = []    genres_col = get_existing_column(        df_clustered,        ["genres_list_parsed", "genres_list", "genres", "genres_clean"]    )    keywords_col = get_existing_column(        df_clustered,        ["keywords_filtered_parsed", "keywords_filtered", "keywords", "keywords_clean"]    )    country_col = get_existing_column(        df_clustered,        ["mainCountry_clean", "mainCountry", "country"]    )    language_col = get_existing_column(        df_clustered,        ["firstLanguage_clean", "firstLanguage", "language"]    )    for cluster_id in sorted(df_clustered["cluster"].unique()):        cluster_df = df_clustered[df_clustered["cluster"] == cluster_id].copy()        top_terms = top_terms_from_cluster_mean(            X_plot_tfidf,            labels,            cluster_id,            tfidf_feature_names,            top_n=top_terms_n        )        genre_counter = Counter()        if genres_col is not None:            for items in cluster_df[genres_col]:                genre_counter.update(safe_parse_list(items))        keyword_counter = Counter()        if keywords_col is not None:            for items in cluster_df[keywords_col]:                keyword_counter.update(safe_parse_list(items))        # Country and language are reported for interpretation only.        dominant_country = "unknown"        if country_col is not None:            mode_vals = cluster_df[country_col].dropna().mode()            if not mode_vals.empty:                dominant_country = mode_vals.iloc[0]        dominant_language = "unknown"        if language_col is not None:            mode_vals = cluster_df[language_col].dropna().mode()            if not mode_vals.empty:                dominant_language = mode_vals.iloc[0]        representative_films = get_representative_films(            matrix=matrix_for_clustering,            model=model,            labels=labels,            cluster_id=cluster_id,            df=df_clustered,            title_col=display_col,            top_n=top_representatives_n        )[display_col].tolist()        profiles.append({            "cluster": int(cluster_id),            "size": int(len(cluster_df)),            "top_terms": ", ".join(top_terms),            "top_genres": ", ".join([g for g, _ in genre_counter.most_common(top_genres_n)]),            "top_keywords": ", ".join([k for k, _ in keyword_counter.most_common(top_keywords_n)]),            "dominant_country (descriptive only)": dominant_country,            "dominant_language (descriptive only)": dominant_language,            "representative_films": " | ".join(representative_films),        })    return pd.DataFrame(profiles).sort_values("cluster").reset_index(drop=True)

In [ ]:
cluster_profiles_k5_df = build_cluster_profiles(    df_clustered=run_k5["df"],    model=run_k5["model"],    matrix_for_clustering=run_k5["X_for_clustering"])cluster_profiles_k6_df = build_cluster_profiles(    df_clustered=run_k6["df"],    model=run_k6["model"],    matrix_for_clustering=run_k6["X_for_clustering"])print("Cluster profiles for k=5")display(cluster_profiles_k5_df)print("Cluster profiles for k=6")display(cluster_profiles_k6_df)

## 6. Select the final clustering solutionReview the evaluation metrics, cluster sizes, and profiles above. The best k may differ from the original notebook now that country/language are excluded — the clusters are purely content-driven, so the thematic separation may shift.**After reviewing**, update `SELECTED_K` below if a different k looks better. By default, we keep k=6 for comparability with the original analysis.

In [ ]:
SELECTED_K = 6  # Update this after reviewing the profiles above.selected_run = run_k6selected_cluster_profiles = cluster_profiles_k6_dfprint(f"Selected final solution: k = {SELECTED_K}")print("Selected cluster sizes:")display(selected_run["cluster_sizes"])print("Selected cluster profiles:")display(selected_cluster_profiles)

## 7. Save clustering outputsResults are saved to `results_content_only/` to keep them separate from the original clustering outputs. The file naming convention is the same so downstream notebooks can switch between the two by changing the results directory path.

In [ ]:
clustered_films_k5_path = RESULTS_DIR / "clustered_films_k5.csv"clustered_films_k6_path = RESULTS_DIR / "clustered_films_k6.csv"cluster_profiles_k5_path = RESULTS_DIR / "cluster_profiles_k5.csv"cluster_profiles_k6_path = RESULTS_DIR / "cluster_profiles_k6.csv"cluster_sizes_k5_path = RESULTS_DIR / "cluster_sizes_k5.csv"cluster_sizes_k6_path = RESULTS_DIR / "cluster_sizes_k6.csv"evaluation_path = RESULTS_DIR / "evaluation_content_only.csv"run_k5["df"].to_csv(clustered_films_k5_path, index=False)run_k6["df"].to_csv(clustered_films_k6_path, index=False)cluster_profiles_k5_df.to_csv(cluster_profiles_k5_path, index=False)cluster_profiles_k6_df.to_csv(cluster_profiles_k6_path, index=False)run_k5["cluster_sizes"].to_csv(cluster_sizes_k5_path, index=False)run_k6["cluster_sizes"].to_csv(cluster_sizes_k6_path, index=False)evaluation_content_only.to_csv(evaluation_path, index=False)print("Saved clustered films k=5 to:", clustered_films_k5_path)print("Saved clustered films k=6 to:", clustered_films_k6_path)print("Saved cluster profiles k=5 to:", cluster_profiles_k5_path)print("Saved cluster profiles k=6 to:", cluster_profiles_k6_path)print("Saved cluster sizes k=5 to:", cluster_sizes_k5_path)print("Saved cluster sizes k=6 to:", cluster_sizes_k6_path)print("Saved evaluation table to:", evaluation_path)

## Output of this notebookThis notebook produces content-only clustering results where country and language **do not influence** cluster membership. Country and language are still reported in the cluster profiles for descriptive purposes.This is the recommended clustering for the admissions predictor, because it allows us to cleanly separate the question "what is this film about?" (cluster) from the question "where was it made?" (country/language as independent variables). In downstream analysis, we can then model how each content cluster performs differently across European markets without circular reasoning.